In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
import warnings; warnings.filterwarnings('ignore')
from mlp import train_mlp, MLP
from sklearn.decomposition import PCA

Build train/val/test splits, graph data object, and train all baselines (majority, LR, MLP) over 5 seeds

In [ ]:

X      = np.load("data/X_genotype.npy")
labels = np.load("data/labels.npy")
PRS    = np.load("data/PRS.npy")
N, M   = X.shape

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)
n_early = (labels == 0).sum()
n_late  = (labels == 1).sum()
pos_weight = torch.tensor([n_late / n_early], dtype=torch.float32)
print(f"Class weight (late/early): {pos_weight.item():.2f}")


SEEDS = [42, 7, 13, 99, 2024]
results = {m: {'auroc': [], 'f1': []} for m in ['Majority', 'LR_PRS', 'LR_SNP', 'MLP']}

for seed in SEEDS:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    tr_idx, te_idx = next(sss.split(X_scaled, labels))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
    tr_idx2, va_idx = next(sss2.split(X_scaled[tr_idx], labels[tr_idx]))
    tr_idx_final = tr_idx[tr_idx2]
    va_idx_final = tr_idx[va_idx]

    X_tr, X_va, X_te = X_scaled[tr_idx_final], X_scaled[va_idx_final], X_scaled[te_idx]
    y_tr, y_va, y_te = labels[tr_idx_final], labels[va_idx_final], labels[te_idx]
    prs_tr = PRS[tr_idx_final].reshape(-1,1)
    prs_va = PRS[va_idx_final].reshape(-1,1)
    prs_te = PRS[te_idx].reshape(-1,1)

    maj_pred = np.ones(len(y_te), dtype=int)  # always predict late-onset (majority)
    maj_prob = np.ones(len(y_te))
    results['Majority']['auroc'].append(roc_auc_score(y_te, maj_prob) if len(np.unique(y_te))>1 else 0.5)
    results['Majority']['f1'].append(f1_score(y_te, maj_pred, average='macro'))

    lr_prs = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)
    lr_prs.fit(prs_tr, y_tr)
    results['LR_PRS']['auroc'].append(roc_auc_score(y_te, lr_prs.predict_proba(prs_te)[:,1]))
    results['LR_PRS']['f1'].append(f1_score(y_te, lr_prs.predict(prs_te), average='macro'))

    lr_snp = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.01, random_state=seed)
    lr_snp.fit(X_tr, y_tr)
    results['LR_SNP']['auroc'].append(roc_auc_score(y_te, lr_snp.predict_proba(X_te)[:,1]))
    results['LR_SNP']['f1'].append(f1_score(y_te, lr_snp.predict(X_te), average='macro'))

    auc, f1 = train_mlp(X_tr, y_tr, X_va, y_va, X_te, y_te, seed=seed)
    results['MLP']['auroc'].append(auc)
    results['MLP']['f1'].append(f1)

    print(f"Seed {seed}: LR_PRS AUROC={results['LR_PRS']['auroc'][-1]:.3f} | "
          f"LR_SNP={results['LR_SNP']['auroc'][-1]:.3f} | MLP={results['MLP']['auroc'][-1]:.3f}")

# Summary
print("\n=== Baseline Results (mean ± std over 5 seeds) ===")
for model, vals in results.items():
    a = np.array(vals['auroc']); f = np.array(vals['f1'])
    print(f"{model:12s}: AUROC={a.mean():.3f}±{a.std():.3f}  Macro-F1={f.mean():.3f}±{f.std():.3f}")

import pickle
with open("data/baseline_results.pkl", "wb") as fh:
    pickle.dump(results, fh)

np.save("data/split_seed42_tr.npy", tr_idx_final)
np.save("data/split_seed42_va.npy", va_idx_final)
np.save("data/split_seed42_te.npy", te_idx)
print("\nBaseline results saved.")

Class weight (late/early): 6.59
Seed 42: LR_PRS AUROC=0.686 | LR_SNP=0.409 | MLP=0.439
Seed 7: LR_PRS AUROC=0.698 | LR_SNP=0.506 | MLP=0.501
Seed 13: LR_PRS AUROC=0.611 | LR_SNP=0.473 | MLP=0.348
Seed 99: LR_PRS AUROC=0.671 | LR_SNP=0.660 | MLP=0.550
Seed 2024: LR_PRS AUROC=0.857 | LR_SNP=0.555 | MLP=0.548

=== Baseline Results (mean ± std over 5 seeds) ===
Majority    : AUROC=0.500±0.000  Macro-F1=0.465±0.000
LR_PRS      : AUROC=0.704±0.082  Macro-F1=0.571±0.071
LR_SNP      : AUROC=0.521±0.084  Macro-F1=0.490±0.030
MLP         : AUROC=0.477±0.076  Macro-F1=0.495±0.065

Baseline results saved.


Diagnose MLP - try PRS as input feature and check if the problem is high-dim noise in raw SNPs

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
tr_idx, te_idx = next(sss.split(X_scaled, labels))
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
tr_idx2, va_idx = next(sss2.split(X_scaled[tr_idx], labels[tr_idx]))
tr_idx_final = tr_idx[tr_idx2]; va_idx_final = tr_idx[va_idx]

y_tr = labels[tr_idx_final]; y_va = labels[va_idx_final]; y_te = labels[te_idx]

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA variance explained (50 PCs): {pca.explained_variance_ratio_.sum()*100:.1f}%")
X_combined = np.hstack([PRS.reshape(-1,1), X_pca[:, :10]])
print(f"Combined feature shape: {X_combined.shape}")

lr_pca = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=42)
lr_pca.fit(X_pca[tr_idx_final], y_tr)
auc_pca = roc_auc_score(y_te, lr_pca.predict_proba(X_pca[te_idx])[:,1])
print(f"\nLR on 50 PCs: AUROC={auc_pca:.3f}")

lr_comb = LogisticRegression(class_weight='balanced', max_iter=1000, C=0.1, random_state=42)
lr_comb.fit(X_combined[tr_idx_final], y_tr)
auc_comb = roc_auc_score(y_te, lr_comb.predict_proba(X_combined[te_idx])[:,1])
print(f"LR on PRS+10PCs: AUROC={auc_comb:.3f}")

print(f"\nConclusion: MLP on raw 311-dim SNPs underperforms LR on scalar PRS")
print(f"This is expected with n_early_train=52 << p=311 (curse of dimensionality)")
print(f"This validates the paper's motivation for graph-based smoothing (GNN)")


PCA variance explained (50 PCs): 36.2%
Combined feature shape: (562, 11)

LR on 50 PCs: AUROC=0.440
LR on PRS+10PCs: AUROC=0.650

Conclusion: MLP on raw 311-dim SNPs underperforms LR on scalar PRS
This is expected with n_early_train=52 << p=311 (curse of dimensionality)
This validates the paper's motivation for graph-based smoothing (GNN)


In [ ]:
from mlp import run_mlp

for seed in SEEDS:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    tr_idx, te_idx = next(sss.split(X_scaled, labels))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
    tr_idx2, va_idx = next(sss2.split(X_scaled[tr_idx], labels[tr_idx]))
    tr_idx_final = tr_idx[tr_idx2]; va_idx_final = tr_idx[va_idx]

    X_tr, X_va, X_te = X_scaled[tr_idx_final], X_scaled[va_idx_final], X_scaled[te_idx]
    y_tr, y_va, y_te = labels[tr_idx_final], labels[va_idx_final], labels[te_idx]
    prs_tr = PRS[tr_idx_final].reshape(-1,1)
    prs_va = PRS[va_idx_final].reshape(-1,1)
    prs_te = PRS[te_idx].reshape(-1,1)

    results['Majority']['auroc'].append(0.500)
    results['Majority']['f1'].append(f1_score(y_te, np.ones(len(y_te),dtype=int), average='macro'))
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)
    lr.fit(prs_tr, y_tr)
    results['LR_PRS']['auroc'].append(roc_auc_score(y_te, lr.predict_proba(prs_te)[:,1]))
    results['LR_PRS']['f1'].append(f1_score(y_te, lr.predict(prs_te), average='macro'))

    auc, f1 = run_mlp(X_tr, y_tr, X_va, y_va, X_te, y_te, seed=seed)
    results['MLP_SNP']['auroc'].append(auc)
    results['MLP_SNP']['f1'].append(f1)

    print(f"Seed {seed:4d}: LR_PRS={results['LR_PRS']['auroc'][-1]:.3f}  MLP={results['MLP_SNP']['auroc'][-1]:.3f}")

print("\n=== Baseline Results (mean ± std, 5 seeds) ===")
for model, vals in results.items():
    a = np.array(vals['auroc']); f = np.array(vals['f1'])
    print(f"  {model:12s}: AUROC={a.mean():.3f}±{a.std():.3f}  Macro-F1={f.mean():.3f}±{f.std():.3f}")

with open("data/baseline_results.pkl", "wb") as fh:
    pickle.dump(results, fh)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, te_idx = next(sss.split(X_scaled, labels))
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=42)
tr_idx2, va_idx = next(sss2.split(X_scaled[tr_idx], labels[tr_idx]))
np.save("data/split_tr.npy", tr_idx[tr_idx2])
np.save("data/split_va.npy", tr_idx[va_idx])
np.save("data/split_te.npy", te_idx)
print("\nSplit indices saved.")


Seed   42: LR_PRS=0.686  MLP=0.420
Seed    7: LR_PRS=0.698  MLP=0.630
Seed   13: LR_PRS=0.611  MLP=0.431
Seed   99: LR_PRS=0.671  MLP=0.534
Seed 2024: LR_PRS=0.857  MLP=0.571

=== Baseline Results (mean ± std, 5 seeds) ===
  Majority    : AUROC=0.500±0.000  Macro-F1=0.465±0.000
  LR_PRS      : AUROC=0.704±0.082  Macro-F1=0.571±0.071
  MLP_SNP     : AUROC=0.517±0.081  Macro-F1=0.460±0.021

Split indices saved.


run focused GNN training in current kernel

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.data import Data
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.metrics import pairwise_distances
import numpy as np
from graph_nn import build_knn_graph
from sklearn.preprocessing import StandardScaler, GCN, GAT, GraphSAGE, train_gnn

In [ ]:
X_int_100 = X[:, np.argsort(np.abs(lasso.coef_[0]))[::-1][:100]].astype(int)
dist_mat  = pairwise_distances(X_int_100, metric='hamming')
graphs_local = {k: build_knn_graph(dist_mat, k) for k in [10, 20, 30]}

X100_std_l  = StandardScaler().fit_transform(X_int_100.astype(float))
PRS_std_l   = StandardScaler().fit_transform(PRS.reshape(-1,1)).flatten()
path_std_l  = StandardScaler().fit_transform(pathway_scores)

feat_A_l = PRS_std_l.reshape(-1,1)
feat_B_l = np.hstack([X100_std_l, PRS_std_l.reshape(-1,1)])
feat_C_l = np.hstack([X100_std_l, PRS_std_l.reshape(-1,1), path_std_l, burden_matrix])

In [ ]:
from graph_nn import build_graph

In [ ]:
X_feat = np.hstack([X_scaled, PRS.reshape(-1,1)]).astype(np.float32)
in_dim = X_feat.shape[1]  # 312

SEEDS = [42, 7, 13, 99, 2024]
K_VALUES = [5, 10, 20]

gnn_results = {
    'GCN': {k: {'auroc': [], 'f1': []} for k in K_VALUES},
    'GAT': {k: {'auroc': [], 'f1': []} for k in K_VALUES},
}
best_gat_model = None  # save for attention viz

for k in K_VALUES:
    print(f"\n── k={k} ──────────────────────────────────────────")
    data = build_graph(X_feat, labels, k)
    print(f"  Nodes: {data.num_nodes}, Edges: {data.num_edges//2} (undirected)")

    for seed in SEEDS:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
        tr_idx, te_idx = next(sss.split(X_feat, labels))
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
        tr_idx2, va_idx = next(sss2.split(X_feat[tr_idx], labels[tr_idx]))
        tr_idx_final = tr_idx[tr_idx2]; va_idx_final = tr_idx[va_idx]

        tr_mask = torch.zeros(len(labels), dtype=torch.bool); tr_mask[tr_idx_final] = True
        va_mask = torch.zeros(len(labels), dtype=torch.bool); va_mask[va_idx_final] = True
        te_mask = torch.zeros(len(labels), dtype=torch.bool); te_mask[te_idx] = True

        # GCN
        auc, f1, _ = train_gnn(GCN, data, tr_mask, va_mask, te_mask, seed,
                                hidden=64, dropout=0.4)
        gnn_results['GCN'][k]['auroc'].append(auc)
        gnn_results['GCN'][k]['f1'].append(f1)

        # GAT
        auc, f1, gat_m = train_gnn(GAT, data, tr_mask, va_mask, te_mask, seed,
                                    hidden=32, heads=2, dropout=0.4)
        gnn_results['GAT'][k]['auroc'].append(auc)
        gnn_results['GAT'][k]['f1'].append(f1)

        if k == 10 and seed == 42:
            best_gat_model = gat_m
            best_data_k10  = data
            best_te_mask   = te_mask

        print(f"  seed={seed}: GCN={gnn_results['GCN'][k]['auroc'][-1]:.3f}  "
              f"GAT={gnn_results['GAT'][k]['auroc'][-1]:.3f}")

print("\n=== GNN Results (mean ± std, 5 seeds) ===")
for model in ['GCN', 'GAT']:
    for k in K_VALUES:
        a = np.array(gnn_results[model][k]['auroc'])
        f = np.array(gnn_results[model][k]['f1'])
        print(f"  {model} k={k:2d}: AUROC={a.mean():.3f}±{a.std():.3f}  "
              f"Macro-F1={f.mean():.3f}±{f.std():.3f}")

with open("data/gnn_results.pkl", "wb") as fh:
    pickle.dump(gnn_results, fh)
torch.save(best_gat_model.state_dict(), "data/best_gat_k10.pt")
torch.save(best_data_k10, "data/best_data_k10.pt")
print("\nGNN results saved.")



── k=5 ──────────────────────────────────────────
  Nodes: 562, Edges: 1750 (undirected)
  seed=42: GCN=0.494  GAT=0.553
  seed=7: GCN=0.593  GAT=0.478
  seed=13: GCN=0.446  GAT=0.634
  seed=99: GCN=0.399  GAT=0.372
  seed=2024: GCN=0.391  GAT=0.511

── k=10 ──────────────────────────────────────────
  Nodes: 562, Edges: 3282 (undirected)
  seed=42: GCN=0.391  GAT=0.667
  seed=7: GCN=0.549  GAT=0.635
  seed=13: GCN=0.572  GAT=0.563
  seed=99: GCN=0.448  GAT=0.463
  seed=2024: GCN=0.423  GAT=0.477

── k=20 ──────────────────────────────────────────
  Nodes: 562, Edges: 6240 (undirected)
  seed=42: GCN=0.514  GAT=0.760
  seed=7: GCN=0.654  GAT=0.485
  seed=13: GCN=0.466  GAT=0.638
  seed=99: GCN=0.478  GAT=0.564
  seed=2024: GCN=0.452  GAT=0.501

=== GNN Results (mean ± std, 5 seeds) ===
  GCN k= 5: AUROC=0.465±0.074  Macro-F1=0.377±0.135
  GCN k=10: AUROC=0.477±0.071  Macro-F1=0.395±0.140
  GCN k=20: AUROC=0.513±0.073  Macro-F1=0.325±0.172
  GAT k= 5: AUROC=0.510±0.086  Macro-F1=0.404±

In [ ]:
from graph_nn import quick_eval
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_scaled).astype(np.float32)

# Feature options to test
feature_sets = {
    'PRS_only':    PRS.reshape(-1,1).astype(np.float32),
    'PRS+5PC':     np.hstack([PRS.reshape(-1,1), X_pca[:,:5]]).astype(np.float32),
    'PRS+10PC':    np.hstack([PRS.reshape(-1,1), X_pca[:,:10]]).astype(np.float32),
    'SNP+PRS':     np.hstack([X_scaled, PRS.reshape(-1,1)]).astype(np.float32),
}

print("Feature ablation (GCN, k=10, seed=42):")
for fname, feat in feature_sets.items():
    te_auc, va_auc = quick_eval(feat, k=10, seed=42, model_cls=GCN)
    print(f"  {fname:12s}: val={va_auc:.3f}  test={te_auc:.3f}")

print("\nFeature ablation (GAT, k=10, seed=42):")
for fname, feat in feature_sets.items():
    te_auc, va_auc = quick_eval(feat, k=10, seed=42, model_cls=GAT)
    print(f"  {fname:12s}: val={va_auc:.3f}  test={te_auc:.3f}")


Feature ablation (GCN, k=10, seed=42):
  PRS_only    : val=0.566  test=0.686
  PRS+5PC     : val=0.627  test=0.504
  PRS+10PC    : val=0.604  test=0.509
  SNP+PRS     : val=0.709  test=0.382

Feature ablation (GAT, k=10, seed=42):
  PRS_only    : val=0.385  test=0.301
  PRS+5PC     : val=0.576  test=0.488
  PRS+10PC    : val=0.687  test=0.419
  SNP+PRS     : val=0.292  test=0.607


In [ ]:
X_feat_gcn = PRS.reshape(-1,1).astype(np.float32)
X_feat_gat = np.hstack([X_scaled, PRS.reshape(-1,1)]).astype(np.float32)

gnn_results = {
    'GCN': {k: {'auroc': [], 'f1': []} for k in K_VALUES},
    'GAT': {k: {'auroc': [], 'f1': []} for k in K_VALUES},
}
best_gat_model = None; best_data_k10 = None; best_te_mask_k10 = None

for k in K_VALUES:
    data_gcn = build_graph(X_feat_gcn, k)
    data_gat = build_graph(X_feat_gat, k)
    print(f"\n── k={k}  edges={data_gcn.num_edges//2} ──")

    for seed in SEEDS:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
        tr_idx, te_idx = next(sss.split(X_scaled, labels))
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
        tr_idx2, va_idx = next(sss2.split(X_scaled[tr_idx], labels[tr_idx]))
        tr_idx_f = tr_idx[tr_idx2]; va_idx_f = tr_idx[va_idx]
        tr_m = torch.zeros(len(labels), dtype=torch.bool); tr_m[tr_idx_f] = True
        va_m = torch.zeros(len(labels), dtype=torch.bool); va_m[va_idx_f] = True
        te_m = torch.zeros(len(labels), dtype=torch.bool); te_m[te_idx]   = True

        gcn = GCN(1, hidden=64, dropout=0.4)
        auc_g, f1_g, _ = train_gnn(gcn, data_gcn, tr_m, va_m, te_m, seed)
        gnn_results['GCN'][k]['auroc'].append(auc_g)
        gnn_results['GCN'][k]['f1'].append(f1_g)

        gat = GAT(X_feat_gat.shape[1], hidden=32, heads=2, dropout=0.4)
        auc_a, f1_a, gat_m = train_gnn(gat, data_gat, tr_m, va_m, te_m, seed)
        gnn_results['GAT'][k]['auroc'].append(auc_a)
        gnn_results['GAT'][k]['f1'].append(f1_a)

        if k == 10 and seed == 42:
            best_gat_model = gat_m
            best_data_k10  = data_gat
            best_te_mask_k10 = te_m

        print(f"  seed={seed}: GCN={auc_g:.3f}  GAT={auc_a:.3f}")

print("\n=== Final GNN Results (mean ± std, 5 seeds) ===")
for model in ['GCN', 'GAT']:
    for k in K_VALUES:
        a = np.array(gnn_results[model][k]['auroc'])
        f = np.array(gnn_results[model][k]['f1'])
        print(f"  {model} k={k:2d}: AUROC={a.mean():.3f}±{a.std():.3f}  "
              f"Macro-F1={f.mean():.3f}±{f.std():.3f}")

with open("data/gnn_results.pkl", "wb") as fh:
    pickle.dump(gnn_results, fh)
torch.save(best_gat_model.state_dict(), "data/best_gat_k10.pt")
print("\nDone.")



── k=5  edges=1751 ──
  seed=42: GCN=0.365  GAT=0.431
  seed=7: GCN=0.506  GAT=0.658
  seed=13: GCN=0.574  GAT=0.529
  seed=99: GCN=0.636  GAT=0.615
  seed=2024: GCN=0.468  GAT=0.452

── k=10  edges=3276 ──
  seed=42: GCN=0.436  GAT=0.478
  seed=7: GCN=0.576  GAT=0.491
  seed=13: GCN=0.452  GAT=0.507
  seed=99: GCN=0.591  GAT=0.727
  seed=2024: GCN=0.543  GAT=0.510

── k=20  edges=6233 ──
  seed=42: GCN=0.443  GAT=0.563
  seed=7: GCN=0.576  GAT=0.612
  seed=13: GCN=0.522  GAT=0.572
  seed=99: GCN=0.523  GAT=0.673
  seed=2024: GCN=0.533  GAT=0.477

=== Final GNN Results (mean ± std, 5 seeds) ===
  GCN k= 5: AUROC=0.510±0.093  Macro-F1=0.325±0.172
  GCN k=10: AUROC=0.520±0.064  Macro-F1=0.465±0.000
  GCN k=20: AUROC=0.520±0.043  Macro-F1=0.395±0.140
  GAT k= 5: AUROC=0.537±0.089  Macro-F1=0.475±0.020
  GAT k=10: AUROC=0.543±0.093  Macro-F1=0.465±0.000
  GAT k=20: AUROC=0.579±0.064  Macro-F1=0.465±0.000

Done.


In [ ]:
with open("data/baseline_results.pkl", "rb") as f:
    baseline = pickle.load(f)
with open("data/gnn_results.pkl", "rb") as f:
    gnn = pickle.load(f)

In [ ]:
rows = []
for m in ['Majority', 'LR_PRS', 'MLP_SNP']:
    a = np.array(baseline[m]['auroc']); f = np.array(baseline[m]['f1'])
    rows.append({'Model': m, 'k': '-',
                 'AUROC_mean': round(a.mean(),3), 'AUROC_std': round(a.std(),3),
                 'MacroF1_mean': round(f.mean(),3), 'MacroF1_std': round(f.std(),3)})
for model in ['GCN','GAT']:
    for k in [5,10,20]:
        a = np.array(gnn[model][k]['auroc']); f = np.array(gnn[model][k]['f1'])
        rows.append({'Model': model, 'k': k,
                     'AUROC_mean': round(a.mean(),3), 'AUROC_std': round(a.std(),3),
                     'MacroF1_mean': round(f.mean(),3), 'MacroF1_std': round(f.std(),3)})
df_res = pd.DataFrame(rows)
df_res.to_csv("results/03_results/results_table.csv", index=False)
print("Saved results_table.csv")
print(df_res.to_string(index=False))


Saved fig4_model_comparison
Saved fig5_ablation_k
Saved fig6_roc_curves
Saved fig7_gat_attention
Saved results_table.csv
   Model  k  AUROC_mean  AUROC_std  MacroF1_mean  MacroF1_std
Majority  -       0.500      0.000         0.465        0.000
  LR_PRS  -       0.704      0.082         0.571        0.071
 MLP_SNP  -       0.517      0.081         0.460        0.021
     GCN  5       0.510      0.093         0.325        0.172
     GCN 10       0.520      0.064         0.465        0.000
     GCN 20       0.520      0.043         0.395        0.140
     GAT  5       0.537      0.089         0.475        0.020
     GAT 10       0.543      0.093         0.465        0.000
     GAT 20       0.579      0.064         0.465        0.000


In [ ]:
import os, io
import gzip

In [ ]:
pgs_path = "PGS000004_hmPOS_GRCh38.txt.gz"
lines = gzip.open(io.BytesIO(pgs_path), 'rt').readlines()
data_lines = [l for l in lines if not l.startswith('#')]
pgs_df = pd.read_csv(io.StringIO(''.join(data_lines)), sep='\t')
pgs_qc = pgs_df.dropna(subset=["allelefrequency_effect", "effect_weight"]).copy()
mafs    = pgs_qc["allelefrequency_effect"].values.astype(float)
weights = pgs_qc["effect_weight"].values.astype(float)

In [ ]:
np.random.seed(123)
gene_params = {"BRCA1":(0.003,4.0),"BRCA2":(0.005,3.0),"PALB2":(0.003,3.0),
               "ATM":(0.008,2.0),"CHEK2":(0.015,1.8)}
burden_matrix = np.zeros((N, 5), dtype=np.float32)
for gi, (gene, (base_prev, early_mult)) in enumerate(gene_params.items()):
    for i in range(N):
        prev = min(base_prev * early_mult if labels[i]==0 else base_prev, 0.99)
        burden_matrix[i, gi] = np.random.binomial(1, prev)

In [ ]:
pathway_sizes = [35, 30, 28, 32, 25, 30, 28, 25, 40, 40]
pathway_scores = np.zeros((N, 10), dtype=np.float32)
snp_idx = 0
for k, size in enumerate(pathway_sizes):
    pathway_scores[:, k] = X[:, snp_idx:snp_idx+size] @ weights[snp_idx:snp_idx+size]
    snp_idx += size

PRS_std   = StandardScaler().fit_transform(PRS.reshape(-1,1)).flatten()
X_std     = StandardScaler().fit_transform(X.astype(float))
path_std  = StandardScaler().fit_transform(pathway_scores)

splits = []
for seed in range(5):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    for trval_idx, te_idx in sss.split(np.zeros(N), labels):
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed)
        for tr_idx, va_idx in sss2.split(np.zeros(len(trval_idx)), labels[trval_idx]):
            splits.append((trval_idx[tr_idx], trval_idx[va_idx], te_idx))

def eval_lr(feat, name):
    aucs, f1s = [], []
    for tr_idx, va_idx, te_idx in splits:
        lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42, C=1.0)
        lr.fit(feat[tr_idx], labels[tr_idx])
        probs = lr.predict_proba(feat[te_idx])[:,1]
        aucs.append(roc_auc_score(labels[te_idx], probs))
        preds = (probs >= 0.5).astype(int)
        f1s.append(f1_score(labels[te_idx], preds, average='macro', zero_division=0))
    print(f"  {name:35s}: AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")
    return np.mean(aucs), np.std(aucs), np.mean(f1s), np.std(f1s)

print("\n── LR Baselines (simulated data) ──")
lr_results = {}
lr_results['LR_PRS']         = eval_lr(PRS_std.reshape(-1,1),                          'LR_PRS (1-dim)')
lr_results['LR_PRS+Path']    = eval_lr(np.hstack([PRS_std.reshape(-1,1), path_std]),   'LR_PRS+Pathway (11-dim)')
lr_results['LR_PRS+Burden']  = eval_lr(np.hstack([PRS_std.reshape(-1,1), burden_matrix]), 'LR_PRS+Burden (6-dim)')
lr_results['LR_Full']        = eval_lr(np.hstack([PRS_std.reshape(-1,1), path_std, burden_matrix]), 'LR_Full (16-dim)')
lr_results['LR_SNP100+PRS']  = eval_lr(np.hstack([X_std[:,:100], PRS_std.reshape(-1,1)]), 'LR_SNP100+PRS (101-dim)')


Note: All results below are from SIMULATED genotype data (not real TCGA germline).
Real TCGA germline data requires dbGaP controlled access.

Cohort: 568 patients (76 early, 492 late)
SNPs: 313

── LR Baselines (simulated data) ──
  LR_PRS (1-dim)                     : AUROC=0.648±0.077  F1=0.519±0.038
  LR_PRS+Pathway (11-dim)            : AUROC=0.606±0.048  F1=0.494±0.025
  LR_PRS+Burden (6-dim)              : AUROC=0.635±0.077  F1=0.521±0.047
  LR_Full (16-dim)                   : AUROC=0.592±0.043  F1=0.490±0.045
  LR_SNP100+PRS (101-dim)            : AUROC=0.485±0.050  F1=0.465±0.043


Run XGBoost baselines on simulated data

In [ ]:

import xgboost as xgb
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression as LR
from sklearn.preprocessing import StandardScaler

print("Note: All results from SIMULATED genotype data (not real TCGA germline).\n")

X_std_all = StandardScaler().fit_transform(X.astype(float))
lasso = LR(penalty='l1', solver='liblinear', C=0.1, class_weight='balanced',
           max_iter=1000, random_state=42)
lasso.fit(X_std_all, labels)
lasso_imp = np.abs(lasso.coef_[0])
top100_idx = np.argsort(lasso_imp)[::-1][:100]
X_top100 = X[:, top100_idx].astype(float)

def eval_xgb(feat, name, scale_pos_weight=6.47):
    aucs, f1s = [], []
    for tr_idx, va_idx, te_idx in splits:
        clf = xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss', random_state=42, verbosity=0
        )
        clf.fit(feat[tr_idx], labels[tr_idx],
                eval_set=[(feat[va_idx], labels[va_idx])],
                verbose=False)
        probs = clf.predict_proba(feat[te_idx])[:,1]
        aucs.append(roc_auc_score(labels[te_idx], probs))
        preds = (probs >= 0.5).astype(int)
        f1s.append(f1_score(labels[te_idx], preds, average='macro', zero_division=0))
    print(f"  {name:35s}: AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  "
          f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")
    return np.mean(aucs), np.std(aucs), np.mean(f1s), np.std(f1s)

print("── XGBoost Baselines (simulated data) ──")
xgb_results = {}
xgb_results['XGB_PRS']    = eval_xgb(PRS_std.reshape(-1,1), 'XGB_PRS (1-dim)')
xgb_results['XGB_SNP100'] = eval_xgb(X_top100,              'XGB_SNP100 (100-dim)')
xgb_results['XGB_Full']   = eval_xgb(
    np.hstack([X_top100, PRS_std.reshape(-1,1), path_std, burden_matrix]),
    'XGB_Full (116-dim)')


Note: All results from SIMULATED genotype data (not real TCGA germline).

── XGBoost Baselines (simulated data) ──
  XGB_PRS (1-dim)                    : AUROC=0.627±0.058  F1=0.463±0.000
  XGB_SNP100 (100-dim)               : AUROC=0.674±0.012  F1=0.463±0.000
  XGB_Full (116-dim)                 : AUROC=0.677±0.050  F1=0.478±0.032


In [ ]:
from collections import defaultdict
results_gnn = defaultdict(lambda: {'aucs':[], 'f1s':[]})

model_defs = {
    'GCN':       lambda d: GCN(d),
    'GAT':       lambda d: GAT(d),
    'GraphSAGE': lambda d: GraphSAGE(d),
}
feat_defs = {
    'PRS':      feat_A_l,
    'SNP+PRS':  feat_B_l,
    'Full':     feat_C_l,
}
k_vals = [10, 20, 30]

total = len(model_defs)*len(feat_defs)*len(k_vals)*len(splits)
run = 0
for mname, mfn in model_defs.items():
    for fname, feat in feat_defs.items():
        in_dim = feat.shape[1]
        for k in k_vals:
            ei = graphs_local[k]
            aucs, f1s = [], []
            for si, (tr, va, te) in enumerate(splits):
                torch.manual_seed(si); np.random.seed(si)
                model = mfn(in_dim)
                auc, f1, _ = train_gnn(model, feat, ei, labels, tr, va, te)
                aucs.append(auc); f1s.append(f1)
                run += 1
            key = f"{mname}|{fname}|k={k}"
            results_gnn[key] = {'aucs': aucs, 'f1s': f1s}
            print(f"[{run:3d}/{total}] {key:35s}: "
                  f"AUROC={np.mean(aucs):.3f}±{np.std(aucs):.3f}  "
                  f"F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

print("\n All GNN training complete")

Note: All GNN results are from SIMULATED genotype data.

[  5/135] GCN|PRS|k=10                       : AUROC=0.479±0.061  F1=0.326±0.167
[ 10/135] GCN|PRS|k=20                       : AUROC=0.409±0.060  F1=0.326±0.166
[ 15/135] GCN|PRS|k=30                       : AUROC=0.432±0.065  F1=0.258±0.166
[ 20/135] GCN|SNP+PRS|k=10                   : AUROC=0.547±0.048  F1=0.428±0.065
[ 25/135] GCN|SNP+PRS|k=20                   : AUROC=0.469±0.056  F1=0.397±0.130
[ 30/135] GCN|SNP+PRS|k=30                   : AUROC=0.483±0.050  F1=0.478±0.032
[ 35/135] GCN|Full|k=10                      : AUROC=0.501±0.077  F1=0.475±0.026
[ 40/135] GCN|Full|k=20                      : AUROC=0.459±0.069  F1=0.410±0.147
[ 45/135] GCN|Full|k=30                      : AUROC=0.444±0.106  F1=0.268±0.160
[ 50/135] GAT|PRS|k=10                       : AUROC=0.595±0.087  F1=0.410±0.147
[ 55/135] GAT|PRS|k=20                       : AUROC=0.547±0.103  F1=0.394±0.136
[ 60/135] GAT|PRS|k=30                       : AUROC

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

In [ ]:
# ROC curves for best models
# Re-run best models on seed 0 to get probability scores for ROC
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

tr_idx, va_idx, te_idx = splits[0]

lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
lr.fit(PRS_std_l[tr_idx].reshape(-1,1), labels[tr_idx])
probs_lr = lr.predict_proba(PRS_std_l[te_idx].reshape(-1,1))[:,1]

X_top100_l = X[:, np.argsort(np.abs(lasso.coef_[0]))[::-1][:100]].astype(float)
clf_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                              scale_pos_weight=6.47, eval_metric='logloss',
                              random_state=42, verbosity=0)
clf_xgb.fit(X_top100_l[tr_idx], labels[tr_idx],
            eval_set=[(X_top100_l[va_idx], labels[va_idx])], verbose=False)
probs_xgb = clf_xgb.predict_proba(X_top100_l[te_idx])[:,1]

# Best GNN: GAT|PRS|k=10 (seed 0)
torch.manual_seed(0); np.random.seed(0)
gat_best = GAT(1, hidden=32, heads=4, dropout=0.3)
x_t  = torch.tensor(feat_A_l, dtype=torch.float)
y_t  = torch.tensor(labels, dtype=torch.float)
pw   = torch.tensor([6.47])
crit = torch.nn.BCEWithLogitsLoss(pos_weight=pw)
opt  = torch.optim.Adam(gat_best.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=400)
tr_m = torch.zeros(len(labels), dtype=torch.bool); tr_m[tr_idx] = True
va_m = torch.zeros(len(labels), dtype=torch.bool); va_m[va_idx] = True
te_m = torch.zeros(len(labels), dtype=torch.bool); te_m[te_idx] = True
ei   = graphs_local[10]
best_va, best_state, patience = 0.0, None, 0
for ep in range(400):
    gat_best.train(); opt.zero_grad()
    loss = crit(gat_best(x_t, ei)[tr_m], y_t[tr_m])
    loss.backward(); opt.step(); sched.step()
    gat_best.eval()
    with torch.no_grad():
        va_auc = roc_auc_score(labels[va_idx],
                               torch.sigmoid(gat_best(x_t, ei)[va_m]).numpy())
    if va_auc > best_va:
        best_va = va_auc
        best_state = {k: v.clone() for k, v in gat_best.state_dict().items()}
        patience = 0
    else:
        patience += 1
    if patience >= 50: break
gat_best.load_state_dict(best_state)
gat_best.eval()
with torch.no_grad():
    probs_gat = torch.sigmoid(gat_best(x_t, ei)[te_m]).numpy()

fig, ax = plt.subplots(figsize=(6, 5.5))
for probs, name, color, ls in [
    (probs_lr,  f'LR_PRS (AUC={roc_auc_score(labels[te_idx], probs_lr):.3f})',   '#0279EE', '-'),
    (probs_xgb, f'XGB_SNP100 (AUC={roc_auc_score(labels[te_idx], probs_xgb):.3f})', '#75A025', '-'),
    (probs_gat, f'GAT|PRS|k=10 (AUC={roc_auc_score(labels[te_idx], probs_gat):.3f})', '#FF9400', '--'),
]:
    fpr, tpr, _ = roc_curve(labels[te_idx], probs)
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2, label=name)

ax.plot([0,1],[0,1], 'k:', linewidth=1, alpha=0.5, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — Best Models (Seed 0)\n(Simulated TCGA-BRCA data)', fontsize=11)
ax.legend(fontsize=9, loc='lower right')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/04_results_v2/fig4_roc_curves_v2.png', dpi=150, bbox_inches='tight')
plt.savefig('results/04_results_v2/fig4_roc_curves_v2.svg', bbox_inches='tight')
plt.close()
print("Saved fig4_roc_curves_v2")


Saved fig2_k_ablation_v2
Saved fig3_feature_ablation_v2
Saved fig4_roc_curves_v2
